# W5C1: A neural classifier, and a competition to improve it

Run every cell from the top. **Everything already works.**

We build the whole classifier together, cell by cell, and it will not be very good. After the break it is yours to improve, in teams, and the scoreboard is public.

Today you will:

1. Turn a document into **one vector** by pooling word embeddings.
2. Build an **MLP** and read every shape in it.
3. Train it and score it with **F1**.
4. Then make it better than it is, against the clock.

Nothing to submit. Answers are in the last cell.

In [ ]:
# Setup. Run this cell first. The corpus downloads once, then it is cached.
import numpy as np
import torch
import torch.nn as nn
import nltk
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

nltk.download("movie_reviews", quiet=True)
from nltk.corpus import movie_reviews

documents = [movie_reviews.raw(f) for f in movie_reviews.fileids()]
labels = [1 if f.startswith("pos") else 0 for f in movie_reviews.fileids()]

print(len(documents), "reviews,", sum(labels), "positive")
print("average length:", int(np.mean([len(d.split()) for d in documents])), "words")
print()
# These are real published reviews, so they are long, informal, and occasionally
# rude. Review 2 is a representative opening.
print(documents[2][:260].replace(chr(10), " "), "...")
print()
print("label:", "positive" if labels[2] else "negative")

In [ ]:
# THREE splits, not two. You tune on validation and you touch test once.
train_docs, test_docs, train_y, test_y = train_test_split(
    documents, labels, test_size=0.25, random_state=0, stratify=labels)
train_docs, val_docs, train_y, val_y = train_test_split(
    train_docs, train_y, test_size=0.20, random_state=0, stratify=train_y)

print(f"train {len(train_docs)}   val {len(val_docs)}   test {len(test_docs)}")
print()
print("Tune on val. The test set is scored once, at the end, by everyone at the")
print("same time. Choosing a model by looking at test is how you fool yourself.")

## Part 1. A document becomes one vector


A network needs a fixed-size vector, and a review is a variable number of words.

Give every word a row of numbers, look up every word in the review, and average
the rows. That average is the document.


<img src="images/embedding-lookup.png" width="640">

In [ ]:
MAX_WORDS = 150          # how much of each review we read
VOCAB_SIZE = 2000        # how many distinct words get their own row


def tokenize(text):
    return text.lower().split()[:MAX_WORDS]


def build_vocabulary(texts, size):
    """Word -> row number. Row 0 is padding, row 1 is anything unknown."""
    from collections import Counter
    counts = Counter(word for text in texts for word in tokenize(text))
    vocabulary = {"<pad>": 0, "<unk>": 1}
    for word, _ in counts.most_common(size):
        vocabulary[word] = len(vocabulary)
    return vocabulary


vocabulary = build_vocabulary(train_docs, VOCAB_SIZE)
print(len(vocabulary), "rows in the vocabulary")
print("the word 'terrible' is row", vocabulary.get("terrible"))

In [ ]:
def encode(texts, vocabulary):
    """Every document becomes MAX_WORDS row numbers, plus a mask saying which
    of those positions are real words rather than padding."""
    ids = np.zeros((len(texts), MAX_WORDS), dtype=np.int64)
    mask = np.zeros((len(texts), MAX_WORDS), dtype=np.float32)
    for i, text in enumerate(texts):
        for j, word in enumerate(tokenize(text)):
            ids[i, j] = vocabulary.get(word, 1)
            mask[i, j] = 1.0
    return torch.from_numpy(ids), torch.from_numpy(mask)


train_ids, train_mask = encode(train_docs, vocabulary)
print("train_ids ", tuple(train_ids.shape), " 1200 reviews x 150 word slots")
print("train_mask", tuple(train_mask.shape), " 1 where there is a real word")
print()
print("the first ten slots of review 0:", train_ids[0][:10].tolist())

In [ ]:
# ================== TRY IT 1 ==================
# `MAX_WORDS` is 150 but the average review is 746 words. What fraction
# of the corpus is the model allowed to read?
# ==============================================


## Part 2. The model


Three layers and nothing else: look up a row per word, average them, then one
hidden layer with a nonlinearity, then two outputs.

Without the nonlinearity, stacking two linear layers just gives you a third
linear layer, and you are back to one straight cut.


<img src="images/mlp-anatomy.png" width="640">

In [ ]:
class Classifier(nn.Module):
    def __init__(self, vocabulary_size, dimensions, hidden):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, dimensions, padding_idx=0)
        self.network = nn.Sequential(
            nn.Linear(dimensions, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 2),
        )

    def forward(self, ids, mask):
        rows = self.embedding(ids) * mask.unsqueeze(-1)   # (batch, words, dims)
        pooled = rows.sum(1) / mask.sum(1, keepdim=True).clamp(min=1)
        return self.network(pooled)                        # (batch, 2)


model = Classifier(len(vocabulary), dimensions=8, hidden=4)

batch_ids, batch_mask = train_ids[:4], train_mask[:4]
print("ids     ", tuple(batch_ids.shape))
print("embedded", tuple(model.embedding(batch_ids).shape), " one row per word")
print("pooled  ", tuple(model.embedding(batch_ids).mean(1).shape), " one row per review")
print("output  ", tuple(model(batch_ids, batch_mask).shape), " one score per class")
print()
print("Trainable numbers:", sum(p.numel() for p in model.parameters()))

In [ ]:
# ================== TRY IT 2 ==================
# The pooling line averages over the words. What happens to the ORDER of
# the review when you do that? Which reviews would that hurt most?
# ==============================================


## Part 3. Train it, and score it


The same four lines as always: clear, measure, blame, step. What is new is how we
judge the result.

**F1** is the harmonic mean of precision and recall, so a model that wins by
always guessing one class scores badly. Accuracy would let that slide.


In [ ]:
def evaluate(model, docs, y, vocabulary):
    """F1 on one split."""
    ids, mask = encode(docs, vocabulary)
    with torch.no_grad():
        predictions = model(ids, mask).argmax(1)
    return f1_score(y, predictions.numpy())


def train(model, ids, mask, y, epochs, learning_rate, batch_size=32, seed=0):
    torch.manual_seed(seed)
    optimiser = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()
    targets = torch.tensor(y)
    for epoch in range(epochs):
        order = torch.randperm(len(targets))
        for start in range(0, len(targets), batch_size):
            batch = order[start:start + batch_size]
            optimiser.zero_grad()                                  # 1. clear
            loss = loss_function(model(ids[batch], mask[batch]),    # 2. measure
                                 targets[batch])
            loss.backward()                                        # 3. blame
            optimiser.step()                                       # 4. step
    return model


torch.manual_seed(0)
model = Classifier(len(vocabulary), dimensions=8, hidden=4)
train(model, train_ids, train_mask, train_y, epochs=3, learning_rate=0.001)

print("validation F1:", round(evaluate(model, val_docs, val_y, vocabulary), 3))


**0.584.** A model that always says "positive" would score 0.667.

That is not a bug, it is a badly configured model: it reads a fifth of each
review, gives every word 8 numbers, and squeezes them through 4 hidden units for
three epochs. Everything about that is yours to change after the break.


In [ ]:
# For comparison, the thing a neural network is supposed to beat.
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

counts = CountVectorizer(max_features=5000)
baseline = LogisticRegression(max_iter=2000)
baseline.fit(counts.fit_transform(train_docs), train_y)

baseline_val = f1_score(val_y, baseline.predict(counts.transform(val_docs)))
print(f"bag of words + logistic regression, validation F1: {baseline_val:.3f}")
print()
print("Counting words beats the neural network by a mile. That is the target.")

In [ ]:
# ================== TRY IT 3 ==================
# Change ONE thing in the training call and see how far it moves the
# validation F1. Which single change helped most?
# ==============================================



---

## Your turn: build a better network

The model below is the smallest one that runs, and it is **not an MLP**. It looks
up a row per word, averages the rows, and puts the result through a single linear
layer. One straight cut, which is where this class started.

Your job is to design something better. **Teams of three, and the scoreboard is
on the board.**

<img src="images/activity-competition.png" width="640">


In [ ]:
# GIVEN. Four settings and one function. You will not need to edit this cell.
MAX_WORDS = 150          # how much of each review is read
VOCAB_SIZE = 2000        # how many distinct words get their own row
EPOCHS = 3               # passes over the training set
LEARNING_RATE = 0.001    # how big each step downhill is


def pool_mean(rows, mask):
    """(batch, words, dims) -> (batch, dims). Average the real words."""
    rows = rows * mask.unsqueeze(-1)
    return rows.sum(1) / mask.sum(1, keepdim=True).clamp(min=1)


def pool_max(rows, mask):
    """(batch, words, dims) -> (batch, dims). Keep the strongest word."""
    return rows.masked_fill(mask.unsqueeze(-1) == 0, -1e9).max(1).values


def glove_rows(vocabulary, dimensions=50):
    """An embedding table started from real GloVe vectors instead of noise."""
    glove = np.load("data/glove-50d-20k.npz")
    known = {str(w): i for i, w in enumerate(glove["words"])}
    table = np.random.normal(0, 0.1, (len(vocabulary), dimensions))
    for word, row in vocabulary.items():
        if word in known:
            table[row] = glove["vectors"][known[word]]
    table[0] = 0
    return torch.tensor(table, dtype=torch.float32)


def train_and_score(build_model, report_on="val", seed=0):
    """Train one architecture and return its F1. Everything is seeded."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    words = build_vocabulary(train_docs, VOCAB_SIZE)
    ids, mask = encode(train_docs, words)

    net = build_model(words)
    train(net, ids, mask, train_y, EPOCHS, LEARNING_RATE, seed=seed)

    docs, y = (val_docs, val_y) if report_on == "val" else (test_docs, test_y)
    return round(evaluate(net, docs, y, words), 3)


print("train_and_score(MyModel) is ready.")


### Ideas

Everything here is a change to the **code**, not to a setting.

1. **Give it a hidden layer.** `nn.Linear` into `nn.ReLU` into `nn.Linear`. This
   is the difference between a straight cut and a boundary that bends, and it is
   the one change the whole first half of the class was about.
2. **Make it wider, or deeper.** Where does adding another layer stop helping?
3. **Change what `forward` does with the words.** `pool_max` instead of
   `pool_mean`, or `torch.cat([pool_mean(...), pool_max(...)], 1)` to keep both.
   Remember what averaging throws away.
4. **Start from real vectors.** `self.embedding.weight.data.copy_(glove_rows(vocabulary))`
   replaces random numbers with GloVe. It needs an embedding of width 50.
5. **Add `nn.Dropout` or `nn.LayerNorm`** between layers.
6. **Change the four settings at the top of the given cell.** One of them is
   worth more than anything on this list. Look at the data to work out which.


In [ ]:
# ================== YOUR TURN 1 ==================
# Rewrite `MyModel` into something better, then run the cell.
#
# Change the layers, the widths, the pooling, whatever you like. The only
# fixed points are the two arguments to `forward` and the two output scores.
#
# Expected: the model as written scores 0.466 on validation. Giving it a
#           hidden layer and a ReLU takes it to 0.634 on its own. Getting near
#           the 0.852 bag-of-words baseline needs several changes together, not
#           one big one.
# =================================================
class MyModel(nn.Module):
    """A word gets 8 numbers, average them, one linear layer to two scores."""

    def __init__(self, vocabulary):
        super().__init__()
        self.embedding = nn.Embedding(len(vocabulary), 8, padding_idx=0)
        self.output = nn.Linear(8, 2)

    def forward(self, ids, mask):
        pooled = pool_mean(self.embedding(ids), mask)
        return self.output(pooled)


print("validation F1:", train_and_score(MyModel))

In [ ]:
# ================== YOUR TURN 2 ==================
# ONE RUN ONLY, when your instructor calls time.
#
# Score your best `MyModel` on the test set. That number goes on the board.
#
# Expected: a test F1 within about 0.02 of your validation number, and
#           sometimes below it. A team whose test score is far below their
#           validation score tuned against validation until it stopped being an
#           honest estimate.
# =================================================
print("TEST F1:", train_and_score(MyModel, report_on="test"))

## Answers

Try each task before reading.

In [ ]:
# TRY IT 1
#   150 / 746, so the model reads about a FIFTH of the average review and never
#   sees the ending, which is where a film review puts its verdict.

# TRY IT 2
#   Averaging throws the order away completely: "not good, actually terrible"
#   and "not terrible, actually good" pool to the same vector. It hurts most on
#   reviews that turn, which is exactly the hard case. pool_max keeps the
#   strongest word instead of diluting it over 150 of them.

# TRY IT 3
#   See the table below. The short answer is that nothing you can change on its
#   own gets close, and that is the point of the exercise.

# YOUR TURN 1
#   The starting model has NO hidden layer, so it is a linear classifier on an
#   averaged embedding: the straight cut this class opened with.
#
#   ONE change at a time, from the starting model:
#
#                                            validation    test
#     the model as given                        0.466      0.472
#     + a hidden layer and a ReLU               0.634      0.634   <- the big one
#     + embedding width 50                      0.626      0.639
#     + LEARNING_RATE 0.01                      0.614      0.613
#     + EPOCHS 30                               0.594      0.617
#     + MAX_WORDS 800                           0.475      0.475
#
#   MAX_WORDS looks useless there. It is not: a linear model on 8 numbers has no
#   capacity to use the extra text. Give the model somewhere to put it and the
#   same change is worth twenty points:
#
#     hidden 64, MAX_WORDS 800, EPOCHS 30, LEARNING_RATE 0.01
#                                               0.832      0.810
#       + width 50, VOCAB_SIZE 20000            0.794      0.816
#       + glove_rows() for the embedding        0.826      0.860   <- best test
#       + cat(pool_mean, pool_max)              0.667      0.667   <- worse!
#       + nn.Dropout(0.3) as well               0.825      0.857
#
#     bag of words + logistic regression        0.852      0.833
#
#   Two honest observations. Concatenating mean and max made things much worse
#   on its own and dropout rescued it, so these choices interact and cannot be
#   judged one at a time. And nothing here beat the bag-of-words baseline on
#   validation: counting words is a genuinely strong method on 1200 documents.
#
#   A reference implementation of the best test score:
#
#     class MyModel(nn.Module):
#         def __init__(self, vocabulary):
#             super().__init__()
#             self.embedding = nn.Embedding(len(vocabulary), 50, padding_idx=0)
#             self.embedding.weight.data.copy_(glove_rows(vocabulary, 50))
#             self.network = nn.Sequential(
#                 nn.Linear(50, 64), nn.ReLU(), nn.Linear(64, 2))
#
#         def forward(self, ids, mask):
#             pooled = pool_mean(self.embedding(ids), mask)
#             return self.network(pooled)
#
#     with MAX_WORDS = 800, VOCAB_SIZE = 20000, EPOCHS = 30, LEARNING_RATE = 0.01.

# YOUR TURN 2
#   Whatever you got. Notice how far validation and test drift apart in the
#   table above: 0.826 against 0.860 for one model, 0.832 against 0.810 for
#   another. A gap of a point or two between two teams is noise, not a winner.

# The three things worth carrying out of today:
#   1. An MLP is a lookup, a pool, and two linear layers with a bend between
#      them. Take the bend out and you have logistic regression, which is
#      exactly what the starting model was.
#   2. Changes interact. MAX_WORDS was worth almost nothing until the model had
#      the capacity to use it, and mean|max pooling needed dropout to help.
#   3. Counting words is a serious baseline. If your neural model cannot beat
#      logistic regression, the neural model is not the answer yet.